In [ ]:
# importing required librabries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# step 01
# If a GPU is listed, Keras will automatically use it during training.
# Check that TensorFlow can see your GPU(s)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

# (Optional) Enable memory growth so TF doesn’t grab all GPU memory at once
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled for GPUs.")
    except RuntimeError as e:
        print("Memory growth error:", e)
else:
    print("No GPU detected, training will run on CPU.")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Memory growth enabled for GPUs.


In [ ]:
# step 02
# loading IMDB sentiment data

vocab_size = 10000      # keep top 10k words

# Load IMDB integer-encoded reviews
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)

In [ ]:
# step 03
# padding and truncating loaded train and test data

max_len = 200           # pad / truncate reviews to 200 tokens

# Pad/truncate sequences so all reviews have the same length (max_len tokens).
# padding="post" adds 0s at the END of shorter sequences (e.g., [1, 2] → [1, 2, 0, 0,...]).
# truncating="post" cuts off EXTRA tokens at the END of longer sequences
# (e.g., [1, 2, 3, 4, 5] → [1, 2, 3] if max_len=3).
# This fixed-length shape is required so we can batch data and feed it into the RNN.
# Pad sequences to fixed length

x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len, padding="post", truncating="post")

x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=max_len, padding="post", truncating="post")

In [ ]:
# step 04
# Creating validation split from test data

# Split 25k test into 15k val + 10k final test
val_size = 15000

x_val = x_test[:val_size]          # first 15k → validation
y_val = y_test[:val_size]

x_test_final = x_test[val_size:]   # remaining 10k → test
y_test_final = y_test[val_size:]

In [ ]:
# step 05
# Building tf.data.Dataset

batch_size = 64

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
val_ds   = tf.data.Dataset.from_tensor_slices((x_val, y_val))
test_ds  = tf.data.Dataset.from_tensor_slices((x_test, y_test))

# Shuffle/batch/prefetch
train_ds = (
    train_ds
    .shuffle(20000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

**Building RNN model**

In [ ]:
# step 06
# building the model

embedding_dim = 64

inputs = keras.Input(shape=(max_len,))

# Learn word embeddings, mask_zero=True makes the model ignore padding tokens
x = layers.Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    mask_zero=True
    )(inputs)

# SimpleRNN processes the sequence left-to-right and keeps a hidden state
x = layers.SimpleRNN(
    128,
    dropout=0.3,            # dropout on inputs to the RNN
    recurrent_dropout=0.3,  # dropout on recurrent connections
    )(x)

# Dense layer with L2 regularization + dropout to fight overfitting
x = layers.Dense(
    64,
    activation="relu",
    kernel_regularizer=keras.regularizers.l2(1e-4),
    )(x)

x = layers.Dropout(0.5)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

rnn_model = keras.Model(inputs, outputs)
rnn_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 64)   │    640,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 200)       │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 128)       │     24,704 │ embedding[0][0],  │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ simple_rnn[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 673,025 (2.57 MB)

 Trainable params: 673,025 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# step 07
# compiling the model

rnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
# step 08
# training the model

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = rnn_model.fit(
    train_ds,
    epochs=15,
    batch_size=64,
    validation_data=val_ds,
    callbacks=[early_stop],
    verbose=2,
)

Epoch 1/15
391/391 - 18s - 46ms/step - accuracy: 0.5014 - loss: 0.7318 - val_accuracy: 0.5024 - val_loss: 0.7018
Epoch 2/15
391/391 - 7s - 19ms/step - accuracy: 0.5028 - loss: 0.7104 - val_accuracy: 0.4944 - val_loss: 0.7009
Epoch 3/15
391/391 - 8s - 19ms/step - accuracy: 0.4954 - loss: 0.7055 - val_accuracy: 0.5003 - val_loss: 0.6992
Epoch 4/15
391/391 - 8s - 20ms/step - accuracy: 0.4997 - loss: 0.7010 - val_accuracy: 0.5031 - val_loss: 0.6989
Epoch 5/15
391/391 - 8s - 19ms/step - accuracy: 0.4998 - loss: 0.6995 - val_accuracy: 0.4975 - val_loss: 0.6983
Epoch 6/15
391/391 - 8s - 19ms/step - accuracy: 0.4976 - loss: 0.6985 - val_accuracy: 0.5025 - val_loss: 0.6980
Epoch 7/15
391/391 - 8s - 19ms/step - accuracy: 0.4981 - loss: 0.6979 - val_accuracy: 0.5027 - val_loss: 0.6972
Epoch 8/15
391/391 - 8s - 19ms/step - accuracy: 0.5002 - loss: 0.6971 - val_accuracy: 0.4965 - val_loss: 0.6970
Epoch 9/15
391/391 - 8s - 20ms/step - accuracy: 0.4964 - loss: 0.6968 - val_accuracy: 0.4998 - val_loss

In [ ]:
# Evaluate the model
test_loss, test_acc = rnn_model.evaluate(test_ds, verbose=0)
print("SimpleRNN IMDB test accuracy:", test_acc)

SimpleRNN IMDB test accuracy: 0.5056800246238708
